In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='7747776d-7e0f-4e8c-af6b-573ae7fc4f1d'),
              AIMessage(content=[{'type': 'text', 'text': "I don’t know what your favorite color is yet, as I don’t have access to your personal history or thoughts unless you’ve told me before!\n\nIf you’d like to share it, I’ll be happy to remember it for our future conversations. Or, if you want me to take a guess, tell me a little bit about yourself and I'll give it a shot!", 'extras': {'signature': 'EnMKcQFpFH0TM/8jdHAiY1G6HBLb5Z+Ahh5PRSFAnh7oesGBNpPom2CaZiBXD6YDWoR7rK9PbbHZOmfZvihCKzIZVNsFwNVxaB1u3vpNnVQCAvtTE3/qEHZvlqWf/148jGduvWDaW4ubcyo6C1+/UucN712y'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f639-93f5-7f11-998f-aab23313a184-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input

At this point, the agent does not have the ability to access the context.
- This is because the context should not be passed to the model directly.
- Instead, it should be passed to the tool calls in an object called `ToolRuntime`, which contains the information that the model has access to, plus a little more.
- We ideally only want to give our model the exact amount of information it needs so that it wont be overloaded with context.

## Accessing Context

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='08d5149a-e0e8-49d7-a883-dd43535fae72'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_590042': 'EnMKcQFpFH0TcbreFj5E4thChwBcv04th13ZFbUAfavEC7L8smzYtqPHNDxLtY0xOeYhJxoC7oo8nY13/GPa/rAk3l+ZbOCgIAwlAn2HvNgyHq/10GvFje7kIx921ISCCWfKyIPZMwBPqfP+3ZVqTDA8caan'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f63c-0a9e-7c21-9c36-4b1fc539e8aa-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_590042', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 13, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='blue', name='get_favou

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green") # The default value in the context could be changed too
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='4de5578b-a752-4d69-8403-978400eff07c'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_621610': 'EnMKcQFpFH0TqdS61rUrbFXAr27QjSEuJMLwWECUm7PJPFbr6otipSA7SAM09CG7g9NFBtcXtE6aoz5BPV3BwYox/cr9XTi7pattdf+r5zo+OOnLtaL+xZ3IfJfGV0WkrqDXD5QP5UyfMsDMnpYkufajz4D7'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f63c-6b4b-7c40-bc6e-bccc01f23eb7-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_621610', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 13, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='green', name='get_favo

Learning Points:

`context_schema` (in `create_agent`) is the type declaration.
You build the agent once, and it has to know ahead of time what shape its runtime context will take. `context_schema=ColourContext` tells it that the context will have `favourite_colour` and `least_favourite_colour`, both strings. This is what lets your tools use ToolRuntime[ColourContext] and read `runtime.context.favourite_colour` with proper type hints and validation. It's much like declaring a function's parameters.

`context=` (in `invoke`) is the actual data.
Each call to the agent can have a different value: `context=ColourContext()` for the defaults, or `context=ColourContext(favourite_colour="green")` for a different user. It's like passing arguments when you call that function. The agent is built once, and the context changes per call or per user.

A useful comparison is with the messages. The messages in invoke are what the model reads. The context is a separate channel of data that never goes to the model unless a tool reads it and returns it. That's why the first example in the notebook can't answer "What is my favourite colour?" correctly: the context exists, but the agent has no tool to access it. After you add `get_favourite_colour`, the tool pulls the value from `runtime.context`, and the model can answer.

So you need both: without `context_schema` the agent doesn't expect or understand a context object, and without `context=` the tools have nothing to read.